# How to configure the neural network

Every inference method in `sbi` trains a neural network. Which network it trains, and with which settings, is described by a **config object** that you pass to the trainer.

There is one config class per model, and it carries only the settings that model actually accepts. A setting the model does not have is therefore not a field on its config, so passing it raises an error right away instead of being silently ignored.

This guide covers:

- picking a model and changing its hyperparameters
- adding an embedding network and controlling z-scoring
- the configs for classifiers, mixed data, and marginal densities
- moving off the deprecated string and factory-function interfaces

## Setup

In [ ]:
import torch

from sbi.utils import BoxUniform

num_dim = 2
prior = BoxUniform(low=-2 * torch.ones(num_dim), high=2 * torch.ones(num_dim))

theta = prior.sample((500,))
x = theta + 1.0 + 0.1 * torch.randn_like(theta)

## Choosing a density estimator

`NPE` and `NLE` take their network as `density_estimator`. Import the config of the model you want and pass an instance:

In [ ]:
from sbi.inference import NPE
from sbi.neural_nets import ZukoNSFConfig

trainer = NPE(prior=prior, density_estimator=ZukoNSFConfig())

Hyperparameters are constructor arguments, so your editor can complete them and type checkers can see them:

In [ ]:
config = ZukoNSFConfig(hidden_features=64, num_transforms=8, num_bins=12)
trainer = NPE(prior=prior, density_estimator=config)

The available density estimator configs are:

| Config | Model |
|---|---|
| `MAFConfig` | masked autoregressive flow (`nflows`) |
| `MAFRQSConfig` | MAF with rational-quadratic splines (`nflows`) |
| `NSFConfig` | neural spline flow (`nflows`) |
| `MADEConfig` | masked autoencoder for density estimation |
| `MDNConfig` | mixture density network |
| `TabPFNConfig` | TabPFN-based estimator |
| `ZukoMAFConfig`, `ZukoNSFConfig`, `ZukoNCSFConfig`, `ZukoNAFConfig`, `ZukoUNAFConfig`, `ZukoBPFConfig`, `ZukoSOSPFConfig`, `ZukoNICEConfig`, `ZukoGFConfig` | the corresponding [`zuko`](https://github.com/probabilists/zuko) flows |

The `nflows` package is no longer maintained, so for new projects we tentatively recommend the `zuko` variants.

`NLE` estimates the likelihood rather than the posterior, but it takes the same configs, because the difference is which variable is modeled and which is conditioned on. That is decided by the trainer, not by the config:

In [ ]:
from sbi.inference import NLE
from sbi.neural_nets import MAFConfig

trainer = NLE(prior=prior, density_estimator=MAFConfig(hidden_features=64))

## Settings a model does not have

`num_bins` is a spline setting. `NSFConfig` has it, `MAFConfig` does not, so the mistake surfaces immediately:

In [ ]:
from sbi.neural_nets import NSFConfig

NSFConfig(num_bins=12)  # fine, a spline flow has bins

try:
    MAFConfig(num_bins=12)
except TypeError as e:
    print(e)

The same applies to a misspelled setting, and to a value outside the allowed set:

In [ ]:
try:
    NSFConfig(hiden_features=64)
except TypeError as e:
    print(e)

try:
    NSFConfig(z_score_input="strucured")
except ValueError as e:
    print(e)

## Inspecting a config

Configs are frozen dataclasses, and their `repr` shows only what you changed, which makes them convenient to log or to put in an experiment record:

In [ ]:
print(ZukoNSFConfig())
print(ZukoNSFConfig(hidden_features=64, num_transforms=8))

Being frozen means a config cannot be edited after construction. To vary one setting, build a new config:

In [ ]:
configs = [ZukoNSFConfig(hidden_features=h) for h in (32, 64, 128)]
print(configs)

## Embedding networks

For high-dimensional data, pass an embedding network that learns summary statistics of the conditioning variable. See [how to use embedding networks](04_embedding_networks.ipynb) for how to choose one.

In [ ]:
from sbi.neural_nets.embedding_nets import FCEmbedding

config = ZukoNSFConfig(embedding_net=FCEmbedding(input_dim=num_dim, output_dim=8))
trainer = NPE(prior=prior, density_estimator=config)

## Z-scoring

Both variables are z-scored independently by default. The standardization is built into the estimator, so you keep passing raw data at inference time.

`z_score_input` applies to the modeled variable and `z_score_condition` to the variable conditioned on. Which is which depends on the method: for `NPE` the input is $\theta$ and the condition is $x$, for `NLE` it is the other way round. Each takes `"independent"` (default), `"structured"`, or `"none"`:

In [ ]:
config = ZukoNSFConfig(z_score_input="independent", z_score_condition="structured")
trainer = NPE(prior=prior, density_estimator=config)

Use `"structured"` when the entries of a variable are not exchangeable, for example a time series, where a single mean and standard deviation across all entries would be the wrong summary. Use `"none"` if the data is already standardized.

## Settings without a field

Some underlying models accept keyword arguments that have no field on the config. `extra_kwargs` forwards them, and marks at the call site that you are stepping outside the checked surface:

In [ ]:
config = ZukoNSFConfig(hidden_features=64, extra_kwargs={"randperm": True})
trainer = NPE(prior=prior, density_estimator=config)

A key that duplicates an existing field is rejected, so there is one place a setting can come from:

In [ ]:
try:
    ZukoNSFConfig(extra_kwargs={"hidden_features": 64})
except ValueError as e:
    print(e)

## Classifiers for NRE

The `NRE` variants train a classifier instead of a density estimator, and take it as `classifier`:

In [ ]:
from sbi.inference import NRE
from sbi.neural_nets import ResNetClassifierConfig

config = ResNetClassifierConfig(hidden_features=64, num_blocks=3)
trainer = NRE(prior=prior, classifier=config)

The classifier configs are `LinearClassifierConfig`, `MLPClassifierConfig`, and `ResNetClassifierConfig`. A classifier sees both variables as inputs rather than conditioning on one, so it takes two embedding networks instead of one:

In [ ]:
config = ResNetClassifierConfig(
    embedding_net_theta=FCEmbedding(input_dim=num_dim, output_dim=8),
    embedding_net_x=FCEmbedding(input_dim=num_dim, output_dim=8),
)

## Mixed data for MNPE and MNLE

`MNPE` and `MNLE` handle data that is partly continuous and partly discrete. `MixedConfig` describes the whole estimator, and the continuous part is configured by nesting that model's own config, so its settings stay validated by its own class:

In [ ]:
from sbi.neural_nets import MixedConfig

config = MixedConfig(
    continuous=ZukoNSFConfig(hidden_features=64, num_transforms=4),
    discrete_hidden_features=32,
)
print(config)

Note that z-scoring of the modeled variable is a property of the continuous model, so it lives on the nested config, while `z_score_condition` stays on `MixedConfig`.

## Marginal densities

`MarginalTrainer` fits an unconditional density, so its configs have no conditioning variable and no `z_score_condition`:

In [ ]:
from sbi.inference.trainers.marginal import MarginalTrainer
from sbi.neural_nets import MarginalNSFConfig

trainer = MarginalTrainer(
    density_estimator=MarginalNSFConfig(hidden_features=64, num_transforms=4)
)

## Building the estimator yourself

The trainer calls `build` on the config once it has seen data, because the network needs the shapes and the z-scoring statistics. You can call it yourself if you want the network outside a trainer, for example for a custom training loop as in [the training interface tutorial](../advanced_tutorials/18_training_interface.ipynb):

In [ ]:
estimator = ZukoNSFConfig(hidden_features=32).build(theta, x)
print(type(estimator).__name__, "| input shape", estimator.input_shape)

The first argument is the modeled variable and the second the one conditioned on, matching `z_score_input` and `z_score_condition`.

## Moving off strings and factory functions

Before the configs, the network was selected with a string, or with one of the `posterior_nn` / `likelihood_nn` / `classifier_nn` factory functions. Both still work; passing a string now warns.

| Before | Now |
|---|---|
| `NPE(prior, density_estimator="nsf")` | `NPE(prior, density_estimator=NSFConfig())` |
| `NPE(prior, density_estimator="zuko_nsf")` | `NPE(prior, density_estimator=ZukoNSFConfig())` |
| `NLE(prior, density_estimator="maf")` | `NLE(prior, density_estimator=MAFConfig())` |
| `NRE(prior, classifier="resnet")` | `NRE(prior, classifier=ResNetClassifierConfig())` |
| `posterior_nn(model="nsf", hidden_features=64)` | `NSFConfig(hidden_features=64)` |
| `likelihood_nn(model="maf", num_transforms=8)` | `MAFConfig(num_transforms=8)` |
| `classifier_nn(model="mlp", hidden_features=64)` | `MLPClassifierConfig(hidden_features=64)` |

The model name moves from a string argument into the class name, and the remaining keyword arguments carry over unchanged.

In [ ]:
from sbi.neural_nets import posterior_nn

# Before
build_fn = posterior_nn(model="zuko_nsf", hidden_features=64, num_transforms=8)
trainer = NPE(prior=prior, density_estimator=build_fn)

# Now
config = ZukoNSFConfig(hidden_features=64, num_transforms=8)
trainer = NPE(prior=prior, density_estimator=config)

One difference worth knowing when you migrate: the factory functions accept settings the chosen model ignores, whereas the config rejects them. If a call that used to run now raises a `TypeError`, that setting was not reaching the network before either.

## See also

- [How to choose neural nets](03_choose_neural_net.ipynb) for which model to pick
- [How to use embedding networks](04_embedding_networks.ipynb)
- [How to choose abstraction levels](24_abstraction_levels.ipynb) for where configs sit relative to the other ways of specifying a network